<a href="https://colab.research.google.com/github/vikrant48/AI-ML-All-AI-related-python-codes-/blob/main/LongChain_ARAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core langchain-groq
!pip install chromadb sentence-transformers pypdf

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]= userdata.get("LANGCHAIN_TRACING_V2")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"   # better for retrieval than MiniLM
)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
#sliding window chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
# this one is just for 1 resume
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/resume1.pdf")
docs = loader.load()

first_page_text = docs[0].page_content

name = extract_name(first_page_text)

for doc in docs:
    doc.metadata["name"] = name
    doc.metadata["years"] = 3   # or extract later

chunks = splitter.split_documents(docs)


In [ ]:
# here for multiple resume
from langchain_community.document_loaders import PyPDFLoader

resume_files = [
    "/resume1.pdf",
    "/resume2.pdf",
    "/resume3.pdf"
]

chunks = []

for file in resume_files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    if not docs:
        print(f"Skipping empty file: {file}")
        continue

    first_page_text = docs[0].page_content
    name = extract_name(first_page_text)
    print("Extracted Name:", name)
    print("file" , file)

    years = 3

    for doc in docs:
        doc.metadata["name"] = name
        doc.metadata["years"] = years
        doc.metadata["source"] = file   # useful later

    chunk = splitter.split_documents(docs)

    chunks.extend(chunk)


In [ ]:

print(f"Total chunks: {len(chunks)}")

In [ ]:
#Embedding + storing
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 10}
)

In [ ]:
query = """
Looking for a Java backend engineer with 3+ years experience,
Spring Boot, AWS, microservices, REST APIs, and database knowledge
"""

In [ ]:
docs = retriever.invoke(query)

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)

for doc in docs:
    grouped[doc.metadata["source"]].append(doc)
    print(doc.metadata["source"])
print(f"Found {len(docs)} documents")

In [ ]:
context_parts = []

for i, doc in enumerate(docs):
    meta = doc.metadata

    context_parts.append(f"""
Candidate {i+1}
Name: {meta.get('name')}
Experience: {meta.get('years')} years
source: {meta.get('source')}

Resume:
{doc.page_content[:1000]}
""")

context = "\n\n----------------\n\n".join(context_parts)

In [ ]:
print(context)

In [ ]:
response = llm.invoke(f"""
Job Query:
{query}

Candidates:
{context}

Instructions:
1. Rank top candidates
2. Show Name
3. Show Match Score (0-100)
4. Show Source (resume name)
5. Give short reason
""")

print(response.content)

now we are using direct method here long chanin LCEL

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chat_history = []

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

Chat History:
{chat_history}

Context:
{context}

Question:
{question}
""")

In [ ]:
def format_chat_history(chat_history):
    print("\n--- CHAT HISTORY ---\n", chat_history)
    return "\n".join([
        f"User: {q}\nAssistant: {a}" for q, a in chat_history
    ])

In [ ]:
chain = (
    {
        "context": retriever,   # automatically does retriever.invoke()
        "question": RunnablePassthrough(),
        "chat_history": lambda x: format_chat_history(chat_history)
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
response = chain.invoke(chain.invoke({"question": "give me all candidate name"}))
print(response)

In [ ]:
ques = "give me all candidate name and count and their skills"
ans = chain.invoke(ques)

print(ans)

chat_history.append((ques, ans))